<img src='https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/tools/frameworks/-/raw/main/img/MSOC_banner.png' align='right' width='100%'/>

<a href="../Index.ipynb">Index</a><br>

**Copyright:** 2026 European Union <br>
**License:** MIT <br>
**Authors:** Ben Loveday (EUMETSAT/Innoflair UG), Hayley Evers-King (EUMETSAT), Juan Ignacio-Gossn (EUMETSAT)

<div class="alert alert-block alert-success">
<h3>Multi-sensor Ocean Colour Course</h3></div>

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>
    
This notebook has the following prerequisites:
- **<a href="https://user.eumetsat.int/register" target="_blank">A EUMETSAT User Portal account</a>** if you want to download Sentinel-3 OLCI marine data from the EUMETSAT Data Store
- **<a href="https://urs.earthdata.nasa.gov/" target="_blank">An Earthdata account</a>** if you want to download PACE OCI, Aqua MODIS or Suomi-VIIRS data from Earthdata.

There are no prerequisite notebooks for this module.
</div>
<hr>

# Mutli sensor validation with the ThoMaS match-up toolkit

### Data used

| Dataset | Source / ID | product description |
|:--------------------:|:-----------------------:|:-------------:|
| Sentinel-3 OLCI level 2 full resolution (operational) | EUMETSAT / EO:EUM:DAT:0407 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:SENTINEL-3:OL_2_WFR___NTC" target="_blank">Description</a> |
| Sentinel-3 OLCI level 2 full resolution (version BC003) reprocessing | EUMETSAT / EO:EUM:DAT:0556 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:0556" target="_blank">Description</a> | EO:EUM:DAT:SENTINEL-3:0556 |
| Sentinel-3 OLCI level 1b full resolution (operational) | EUMETSAT / EO:EUM:DAT:0409 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:SENTINEL-3:OL_1_EFR___NTC" target="_blank">Description</a> |
| Sentinel-3 OLCI level 1b full resolution (version BC004) reprocessing| EUMETSAT / EO:EUM:DAT:0885 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:0885" target="_blank">Description</a> |
| PACE OCI Level-2 Regional Apparent Optical Properties - Near Real-time (NRT) Data, version 3.0 | PACE_OCI_L2_AOP | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3620139598-OB_CLOUD.html" target="_blank">Description</a> |
| Aqua MODIS Regional Ocean Color (OC) Data, version R2022.0 | Earthdata / MODISA_L2_OC | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3380708980-OB_CLOUD.html" target="_blank">Description</a> |
| NOAA-20 VIIRS Level-2 Regional Ocean Color (OC) - Near Real-time (NRT) Data, version 2022.0 | Earthdata / VIIRSN_L2_OC | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3396928895-OB_CLOUD.html" target="_blank">Description</a> |

### Learning outcomes

At the end of this notebook you will know;
* how to use the ThoMaS toolkit to perform multisensor match-up validation extractions and analyses

### Outline

ThoMaS can be used to perform match-up analyses against multiple sensors. In this notebook, we will expand on the previous <a href="./Single_sensor_match_up_analysis.ipynb" target="_blank">single sensor</a> notebook to perform a multi-sensor analysis that includes Sentinel-3 OLCI, NASA PACE OCI and VIIRS NOAA-20.

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing all of the libraries that we need to run this notebook. If you have built your python environment (`msoc`) using the file provided in this repository, then you should have everything you need. For more information on building environment, please see the repository **<a href="../README.md" target="_blank">README</a>**.

In [1]:
import os                                   # a library that allows us access to basic operating system commands like making directories
import sys                                  # a library that provides access to system commands
import pandas as pd                         # a library that helps us manipulate data 
import shutil                               # a library that allows us access to basic operating system commands like copy
import numpy as np                          # a library that provides support for array-based mathematics
import glob                                 # a library that helps us find files
import image_carousel                       # a bespoke function to make an image carousel
from pathlib import Path                    # a library to help us manage paths

Everything is now imported, we can proceed...

<div class="alert alert-warning" role="alert">

## Defining functions

</div>

Before we go any further we are going to define a quick function that helps us to write our configuration options to a file.

*Note: We don't need to write our configurations in python, we could always just write the configuration file directly and point ThoMaS to it.*

In [2]:
# Write config_params sections into config_file.ini
def write_config_file(path_to_config_file,config_params):
    with open(path_to_config_file, 'w') as text_file:
        for section,section_params in config_params.items():
            text_file.write('\n[%s]\n' % (section))
            for param, value in section_params.items():
                text_file.write('%s: %s\n' % (param, value))

In [3]:
def find_git_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path] + list(path.parents):
        if (parent / ".git").exists():
            return parent
    return None

<div class="alert alert-info" role="alert">

## Configuring the ThoMaS toolkit

</div>

The **ThoMaS** toolkit is included as a submodule in this repository. To import it, we just need to make sure that Python can find the correct path to it. Lets find and import ThoMaS below.

In [4]:
thomas_path = os.path.join(find_git_root(), "3_courses", "msoc", "submodules", "ThoMaS")
sys.path.append(thomas_path)
from main import ThoMaS_main as ThoMaS

Please see the <a href="./Single_sensor_match_up_analysis.ipynb" target="_blank">single sensor</a> notebook, if you need more help with this.

<div class="alert alert-info" role="alert">

## Example: Full match-up exercise with *in situ* data from an AERONET-OC site (adapted from <a href="https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS/-/blob/main/README_examples.md?ref_type=heads" target="_blank">ThoMaS example 9</a>)
[Back to top](#Contents)

</div>

<div class="alert alert-success" role="alert">

Run ThoMaS for:

1. full matchup exercise: satellite extractions + minifiles + extraction statistics + matchup statistics
1. download insitu data from AERONET-OC station Casablanca_Platform and convert to SeaBASS/OCDB format
1. apply <a href="https://ioccg.org/bibliography/morel-a-antoine-d-and-b-gentili-2002/" target="_blank">Morel, Antoine and Gentili, 2002</a>
1. EUMETSAT standard protocol for extractions and for insitu-extraction comparison.

</div>

#### Description

<hr>

In this final example we are going to perform a full analysis using a subset of data from the MOBY ocean colour database as our *in situ* data.

1. full matchup exercise: satellite extractions + minifiles + extraction statistics + matchup statistics
1. download insitu data from AERONET-OC station Soceongcho and convert to SeaBASS/OCDB format
1. EUMETSAT standard protocol for extractions and for insitu-extraction comparison.
1. Run ThoMaS with a subset of AERONET-OC data belonging to the Casablanca_Platform
station (PI Marco Talone and Frederic Melin) during March-July 2024, and matching with EUMETSAT-processed OLCI A and
l2gen-processed NOAA-20/VIIRS and PACE/OCI data.
1. I am interested in using AERONET-OC data of quality level 1.5.
1. I want to test the impact on matchups by using window sizes 3x3 and 5x5 and by using different extraction protocols: <a href="https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS/-/blob/main/protocols/l2gen_L2_PACE_OCI/EUMETSAT_standard_L2" target="_blank">EUMETSAT's standard protocol</a> and <a href="https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS/-/blob/main/protocols/l2gen_L2_PACE_OCI/Bailey_and_Werdell_2006" target="_blank">Bailey and Werdell 2006</a> protocol.
<hr>

First off, lets create a directory to store our experiment in.

In [5]:
# Define and create your main output path for this example:
example_tag = "Casablanca_Platform" 

output_path = os.path.join(os.getcwd(), example_tag)
if not os.path.exists(output_path):
    os.mkdir(output_path)

And, as in the <a href="./Single_sensor_match_up_analysis.ipynb" target="_blank">single sensor</a> example, we will build our configuration file. This time, however, we need to make some adaptations to ensure that we use the remote AERONET-OC data as our *in situ* validation source.

In [6]:
# Build your config_file.ini
path_to_config_file = os.path.join(output_path, 'config_file.ini')
config_params = {}

# global
config_params['global'] = {}
config_params['global']['path_output'] = output_path
config_params['global']['SetID'] = 'Casablanca_Platform'

# workflow
config_params['workflow'] = {}
config_params['workflow']['workflow'] = 'insitu, SatData, minifiles, EDB, MDB'

# AERONETOC
config_params['AERONETOC'] = {}
config_params['AERONETOC']['AERONETOC_pathNative'] = os.path.join(output_path, 'AERONETOC_native')
config_params['AERONETOC']['AERONETOC_dateStart'] = '2024-04-01T00:00:00'
config_params['AERONETOC']['AERONETOC_dateEnd'] = '2024-04-30T00:00:00'
config_params['AERONETOC']['AERONETOC_dataLevel'] = 1.5
config_params['AERONETOC']['AERONETOC_station'] = 'Casablanca_Platform'

# insitu
config_params['insitu'] = {}
config_params['insitu']['insitu_data2OCDBfile'] = "AERONETOC"
config_params['insitu']['insitu_input'] = os.path.join(output_path, 'Casablanca_Platform.sb')
config_params['insitu']['insitu_satellite_time_tolerance_seconds'] = 3600
config_params['insitu']['insitu_getAncillary'] = False 
config_params['insitu']['insitu_BRDF'] = 'M02'

# satellite
config_params['satellite'] = {}
config_params['satellite']['satellite_path-to-SatData'] = os.path.join(output_path, 'SatData')
config_params['satellite']['satellite_source'] = 'online'
config_params['satellite']['satellite_platforms'] = 'S3, PACE, NOAA-20'
config_params['satellite']['satellite_resolutions'] = 'FR'
config_params['satellite']['satellite_BRDF'] = 'M02'

# minifiles
config_params['minifiles'] = {}
config_params['minifiles']['minifiles_winSize'] = 5

# EDB
config_params['EDB'] = {}
config_params['EDB']['EDB_protocols_L2'] = 'EUMETSAT_standard_L2, Bailey_and_Werdell_2006'
config_params['EDB']['EDB_winSizes'] = "3, 5"

# MDB
config_params['MDB'] = {}
config_params['MDB']['MDB_time-interpolation'] = 'insitu2satellite_NN'
config_params['MDB']['MDB_stats_plots'] = True
config_params['MDB']['MDB_stats_MonteCarlo'] = 100
config_params['MDB']['MDB_stats_protocol'] = 'EUMETSAT_standard_L2'

# Write config_params sections into config_file.ini
write_config_file(path_to_config_file, config_params)

...and run!

In [7]:
# Run ThoMaS and check all the outputs in the output directory
ThoMaS(path_to_config_file)


Running Casablanca_Platform

Building satellite_datasets file from specified options in config_file...

Step insitu
Creating IDB (in situ database) netcdf file and calculating timeRanges and satellite datasets from input SeaBASS/OCDB file (in situ)
insitu: Spectral matching for sensor OCI and platform PACE
AERONET-OC: Extracting .zip
Sanitised file successfully saved: /Users/benloveday/Code/git_repositories/CMTS/internal/ocean/applications/ocean-colour-applications/3_courses/msoc/Day5/Casablanca_Platform/Casablanca_Platform_sanitised.sb
Building insitu netCDF file for insitu set Casablanca_Platform, spectrally matched to platform PACE, sensor OCI
Computing spectral reconstruction method from Talone, Zibordi and Pitarch
In situ uncertainties propagated through spectral match scheme via simple Monte Carlo approach
insitu: Spectral matching for sensor VIIRS and platform NOAA-20
Already exists!: /Users/benloveday/Code/git_repositories/CMTS/internal/ocean/applications/ocean-colour-applicat

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

NASA Earthaccess, downloading ... 


...

NASA Earthaccess, downloading ... 


QUEUEING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2 [00:00<?, ?it/s]

Writing SatDataList for set NOAA-20_FAUNS_VIIRS_L2_l2gen_OBPG_FR_Casablanca_Platform to: /Users/benloveday/Code/git_repositories/CMTS/internal/ocean/applications/ocean-colour-applications/3_courses/msoc/Day5/Casablanca_Platform/SatDataLists/SatDataList_NOAA-20_FAUNS_VIIRS_L2_l2gen_OBPG_FR_Casablanca_Platform.txt

Step minifiles
Extracting minifiles from satellite data
Creating minifile: PACE_OCI.20240401T123447.L2.OC_AOP.V3_1.COMBE_E00135800N4071700_5x5.nc
Creating minifile: PACE_OCI.20240403T120633.L2.OC_AOP.V3_1.COMBE_E00135800N4071700_5x5.nc
Creating minifile: PACE_OCI.20240404T124136.L2.OC_AOP.V3_1.COMBE_E00135800N4071700_5x5.nc
Creating minifile: PACE_OCI.20240409T122008.L2.OC_AOP.V3_1.COMBE_E00135800N4071700_5x5.nc
Creating minifile: PACE_OCI.20240410T111650.L2.OC_AOP.V3_1.COMBE_E00135800N4071700_5x5.nc
Creating minifile: PACE_OCI.20240411T115152.L2.OC_AOP.V3_1.COMBE_E00135800N4071700_5x5.nc
Creating minifile: PACE_OCI.20240412T122654.L2.OC_AOP.V3_1.COMBE_E00135800N4071700_5x5.nc

...and check our outputs.

In [8]:
image_paths = glob.glob(os.path.join(os.getcwd(), example_tag, "MDB", "plots_global", "*.png"))
carousel = image_carousel.create_image_carousel(image_paths, height=800, width=1200)
carousel

<div class="alert alert-success" role="alert">

## What next?

If you have validation data in your area of interest, either from the AERNOTET-OC network, the SeaBass or OCDB databases or your own campaigns, explore the above parameters to see how ThomaS can be applied to your work. If you need guidance on field campaigns for your studies, or want to learn more aboue EUMETSAT Fiducial Reference Measurement (FRM) validation measurements and protocols, please talk to us!

</div>

You can find many more example of configuration files in the ./ThoMaS/examples directory. Building on these, and the examples shown above you should be able to build a validation workflow that works for your circumstances!

<hr>
<a href="../Index.ipynb">Index</a>
<hr>
<a href="https://gitlab.eumetsat.int/eumetlab/ocean">View on GitLab</a> | <a href="https://training.eumetsat.int/">EUMETSAT Training</a> | <a href=mailto:ops@eumetsat.int>Contact helpdesk for support </a> | <a href=mailto:.training@eumetsat.int>Contact our training team to collaborate on and reuse this material</a></span></p>